In [ ]:
# ! pip install scikit-learn scipy matplotlib

In [31]:
import json
import numpy as np
import os

def update_labels_with_cdf(file_paths, overwrite=False):
    all_data = {filepath: [] for filepath in file_paths}
    all_flow_sizes = []

    print(f"\n[{file_paths[0].split('/')[2] if len(file_paths) > 0 else '알 수 없음'}] 데이터셋 처리 시작...")
    
    # 1. 파일에서 데이터 읽기
    for filepath in file_paths:
        if not os.path.exists(filepath):
            print(f"  -> ⚠️ 경고: {filepath} 파일이 존재하지 않아 건너뜁니다.")
            continue
            
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                data = json.loads(line.strip())
                all_data[filepath].append(data)
                all_flow_sizes.append(data['flow_size_bytes'])

    total_samples = len(all_flow_sizes)
    if total_samples == 0:
        print("  -> ❌ 처리할 데이터가 없습니다.")
        return

    # 2. 통합된 CDF 계산
    unique_sizes, counts = np.unique(all_flow_sizes, return_counts=True)
    cdf_values = np.cumsum(counts) / total_samples
    size_to_cdf_map = dict(zip(unique_sizes, cdf_values))

    # 3. 라벨 업데이트 및 저장
    for filepath, data_list in all_data.items():
        if not data_list:
            continue
            
        # 덮어쓰기 여부에 따라 저장할 파일 경로 설정
        if overwrite:
            save_filepath = filepath
        else:
            base, ext = os.path.splitext(filepath)
            save_filepath = f"{base}_with_cdf{ext}"
        
        # 만약 저장할 폴더가 없다면 생성 (보통 존재하겠지만 방어적 코드)
        os.makedirs(os.path.dirname(save_filepath), exist_ok=True)
        
        with open(save_filepath, 'w', encoding='utf-8') as f:
            for data in data_list:
                data['label'] = float(size_to_cdf_map[data['flow_size_bytes']])
                f.write(json.dumps(data) + '\n')
                
        print(f"  -> ✅ 완료: {save_filepath} 저장됨.")


# === 실행 부분 ===
dataset_types = ("dctcp", "fb", "vl2", "univ1")

for d_type in dataset_types:
    target_files = [
        f"dataset/elephant_dst_to_src/{d_type}/seq3/dataset.jsonl", 
        f"dataset/elephant_dst_to_src/{d_type}/seq5/dataset.jsonl", 
        f"dataset/elephant_dst_to_src/{d_type}/seq10/dataset.jsonl"
    ]

    # overwrite=False (기본값): dataset_with_cdf.jsonl 로 새로 저장
    # overwrite=True : 기존 dataset.jsonl 파일에 그대로 덮어쓰기
    update_labels_with_cdf(target_files, overwrite=False)
    
print("모든 데이터셋 처리가 완료되었습니다!")


[dctcp] 데이터셋 처리 시작...
  -> ✅ 완료: dataset/elephant_dst_to_src/dctcp/seq3/dataset_with_cdf.jsonl 저장됨.
  -> ✅ 완료: dataset/elephant_dst_to_src/dctcp/seq5/dataset_with_cdf.jsonl 저장됨.
  -> ✅ 완료: dataset/elephant_dst_to_src/dctcp/seq10/dataset_with_cdf.jsonl 저장됨.

[fb] 데이터셋 처리 시작...
  -> ✅ 완료: dataset/elephant_dst_to_src/fb/seq3/dataset_with_cdf.jsonl 저장됨.
  -> ✅ 완료: dataset/elephant_dst_to_src/fb/seq5/dataset_with_cdf.jsonl 저장됨.
  -> ✅ 완료: dataset/elephant_dst_to_src/fb/seq10/dataset_with_cdf.jsonl 저장됨.

[vl2] 데이터셋 처리 시작...
  -> ✅ 완료: dataset/elephant_dst_to_src/vl2/seq3/dataset_with_cdf.jsonl 저장됨.
  -> ✅ 완료: dataset/elephant_dst_to_src/vl2/seq5/dataset_with_cdf.jsonl 저장됨.
  -> ✅ 완료: dataset/elephant_dst_to_src/vl2/seq10/dataset_with_cdf.jsonl 저장됨.

[univ1] 데이터셋 처리 시작...
  -> ✅ 완료: dataset/elephant_dst_to_src/univ1/seq3/dataset_with_cdf.jsonl 저장됨.
  -> ✅ 완료: dataset/elephant_dst_to_src/univ1/seq5/dataset_with_cdf.jsonl 저장됨.
  -> ✅ 완료: dataset/elephant_dst_to_src/univ1/seq10/dataset_with_cdf

In [32]:
import json
import random
from pathlib import Path

for d_type in ("dctcp", "fb","vl2", "univ1"):
    for s_type in ("seq3", "seq5", "seq10"):
        # 원본 jsonl 파일 경로
        input_path = Path(f"dataset/elephant_dst_to_src/{d_type}/{s_type}/dataset_with_cdf.jsonl")

        # 저장 경로
        train_output_path = input_path.parent / "train.jsonl"
        test_output_path = input_path.parent / "test.jsonl"

        # 랜덤 시드 고정 (재현 가능)
        random.seed(42)

        # jsonl 읽기
        with open(input_path, "r", encoding="utf-8") as f:
            lines = f.readlines()

        # 데이터 개수 확인
        total_count = len(lines)

        # test 개수
        test_size = 1000

        if total_count < test_size:
            raise ValueError(f"데이터 개수({total_count})가 test_size({test_size})보다 작습니다.")

        # 셔플
        random.shuffle(lines)

        # 분할
        test_lines = lines[:test_size]
        train_lines = lines[test_size:]

        # train 저장
        with open(train_output_path, "w", encoding="utf-8") as f:
            f.writelines(train_lines)

        # test 저장
        with open(test_output_path, "w", encoding="utf-8") as f:
            f.writelines(test_lines)

        print(f"전체 데이터 수 : {total_count}")
        print(f"Train 데이터 수: {len(train_lines)}")
        print(f"Test 데이터 수 : {len(test_lines)}")

        print(f"\n저장 완료")
        print(f"Train 파일: {train_output_path}")
        print(f"Test 파일 : {test_output_path}")

전체 데이터 수 : 9600
Train 데이터 수: 8600
Test 데이터 수 : 1000

저장 완료
Train 파일: dataset\elephant_dst_to_src\dctcp\seq3\train.jsonl
Test 파일 : dataset\elephant_dst_to_src\dctcp\seq3\test.jsonl
전체 데이터 수 : 9600
Train 데이터 수: 8600
Test 데이터 수 : 1000

저장 완료
Train 파일: dataset\elephant_dst_to_src\dctcp\seq5\train.jsonl
Test 파일 : dataset\elephant_dst_to_src\dctcp\seq5\test.jsonl
전체 데이터 수 : 9600
Train 데이터 수: 8600
Test 데이터 수 : 1000

저장 완료
Train 파일: dataset\elephant_dst_to_src\dctcp\seq10\train.jsonl
Test 파일 : dataset\elephant_dst_to_src\dctcp\seq10\test.jsonl
전체 데이터 수 : 9600
Train 데이터 수: 8600
Test 데이터 수 : 1000

저장 완료
Train 파일: dataset\elephant_dst_to_src\fb\seq3\train.jsonl
Test 파일 : dataset\elephant_dst_to_src\fb\seq3\test.jsonl
전체 데이터 수 : 9600
Train 데이터 수: 8600
Test 데이터 수 : 1000

저장 완료
Train 파일: dataset\elephant_dst_to_src\fb\seq5\train.jsonl
Test 파일 : dataset\elephant_dst_to_src\fb\seq5\test.jsonl
전체 데이터 수 : 9600
Train 데이터 수: 8600
Test 데이터 수 : 1000

저장 완료
Train 파일: dataset\elephant_dst_to_src\fb\seq10\trai

In [26]:
for d_type in ("vl2", "univ1"):
    for s_type in ("seq3", "seq5", "seq10"):
        # 원본 jsonl 파일 경로
        train_path = Path(f"dataset/elephant_dst_to_src/{d_type}/{s_type}/train.jsonl")

        # 출력 파일
        output_path = train_path.parent / "train_balanced.jsonl"

# 데이터 읽기
        data = []

        with open(train_path, "r", encoding="utf-8") as f:
            for line in f:
                data.append(json.loads(line))

        # label 기준 분리
        label_0 = [d for d in data if d["label"] == 0]
        label_1 = [d for d in data if d["label"] == 1]

        print(f"label 0 count: {len(label_0)}")
        print(f"label 1 count: {len(label_1)}")

        # 부족한 수 계산
        target_count = len(label_0)

        if len(label_1) < target_count:
            # 복원 추출(중복 허용)
            additional = random.choices(label_1, k=target_count - len(label_1))

            balanced_label_1 = label_1 + additional
        else:
            balanced_label_1 = random.sample(label_1, target_count)

        # 합치기
        balanced_data = label_0 + balanced_label_1

        # 셔플
        random.shuffle(balanced_data)

        # 저장
        with open(output_path, "w", encoding="utf-8") as f:
            for item in balanced_data:
                f.write(json.dumps(item) + "\n")

        print(f"\nBalanced dataset saved.")
        print(f"Total samples: {len(balanced_data)}")
        print(f"Saved to: {output_path}")

label 0 count: 0
label 1 count: 1

Balanced dataset saved.
Total samples: 0
Saved to: dataset\elephant_dst_to_src\vl2\seq3\train_balanced.jsonl
label 0 count: 0
label 1 count: 1

Balanced dataset saved.
Total samples: 0
Saved to: dataset\elephant_dst_to_src\vl2\seq5\train_balanced.jsonl
label 0 count: 0
label 1 count: 1

Balanced dataset saved.
Total samples: 0
Saved to: dataset\elephant_dst_to_src\vl2\seq10\train_balanced.jsonl
label 0 count: 0
label 1 count: 1

Balanced dataset saved.
Total samples: 0
Saved to: dataset\elephant_dst_to_src\univ1\seq3\train_balanced.jsonl


KeyboardInterrupt: 

In [37]:
import json
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from models import DynamicPacketGRU

# ---------------------------------------------------------
# 1. Dataset 및 Collate 함수 (패딩 없음)
# ---------------------------------------------------------
class ExactPacketDataset(Dataset):
    def __init__(self, data_list):
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        x = torch.tensor(item["x"], dtype=torch.float32)
        label = torch.tensor([item["label"]], dtype=torch.float32)
        return x, label

def collate_fn_no_pad(batch):
    xs = torch.stack([item[0] for item in batch])
    labels = torch.stack([item[1] for item in batch])
    return xs, labels

# ---------------------------------------------------------
# 2. 데이터 불러오기 함수 (전체 데이터 학습용)
# ---------------------------------------------------------
def load_data(original_path):
    all_data = []
    with open(original_path, 'r', encoding='utf-8') as f:
        for line in f:
            all_data.append(json.loads(line))
            
    # 셔플 (데이터 편향 방지 및 재현성을 위해 시드 고정)
    random.seed(42)
    random.shuffle(all_data)
    
    print(f"  [안내] 학습 데이터 개수: {len(all_data)}개")
    return all_data

# ---------------------------------------------------------
# 3. 학습 핵심 함수 (테스트 제거)
# ---------------------------------------------------------
def train_model(model, train_loader, optimizer, criterion, device, epochs=5):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for x_batch, label_batch in train_loader:
            x_batch, label_batch = x_batch.to(device), label_batch.to(device)
            batch_size = x_batch.size(0)
            
            direction_idx = torch.ones(batch_size, dtype=torch.long).to(device)
            
            optimizer.zero_grad()
            outputs = model(x_batch, direction_idx=direction_idx, enable_early_exit=False)
            
            final_preds = outputs[:, -1, :] 
            loss = criterion(final_preds, label_batch)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        print(f"    Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

# ---------------------------------------------------------
# 4. 실질적인 순차적(Curriculum) 학습 실행 파트
# ---------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DynamicPacketGRU(input_size=18, hidden_size=64).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

dataset_type = "fb"

# 원본 데이터 파일 경로 매핑
dataset_files = {
    3: f"dataset/elephant_dst_to_src/{dataset_type}/seq3/train.jsonl",
    5: f"dataset/elephant_dst_to_src/{dataset_type}/seq5/train.jsonl",
    10: f"dataset/elephant_dst_to_src/{dataset_type}/seq10/train.jsonl"
}

print("=== 커리큘럼 전체 데이터 학습 및 저장 파이프라인 시작 ===")
for seq_len, src_path in dataset_files.items():
    print(f"\n--------------------------------------------------")
    print(f"[Phase] 시퀀스 길이 {seq_len} 단계 시작")
    print(f"--------------------------------------------------")
    
    # 가중치 파일명 정의
    weight_save_path = f"dataset/elephant_dst_to_src/{dataset_type}/seq{seq_len}/weights.pt"
    
    try:
        # 1. 데이터 불러오기 (전체 데이터)
        train_list = load_data(src_path)
        
        # 2. DataLoader 생성
        train_dataset = ExactPacketDataset(train_list)
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn_no_pad)
        
        # 3. 모델 학습 (가중치가 누적되며 이어서 학습됨)
        train_model(model, train_loader, optimizer, criterion, device, epochs=5)
        
        # 4. 현재 단계의 가중치 저장
        torch.save(model.state_dict(), weight_save_path)
        print(f"  [성공] 시퀀스 {seq_len} 학습 완료. 가중치가 '{weight_save_path}'에 저장되었습니다.")
        
    except FileNotFoundError:
        print(f"  [오류] {src_path} 파일을 찾을 수 없어 이 단계를 건너넙니다.")

print("\n=== 모든 단계의 학습 및 파일 저장이 완료되었습니다. ===")

=== 커리큘럼 전체 데이터 학습 및 저장 파이프라인 시작 ===

--------------------------------------------------
[Phase] 시퀀스 길이 3 단계 시작
--------------------------------------------------
  [안내] 학습 데이터 개수: 8600개
    Epoch [1/5], Loss: 0.6763
    Epoch [2/5], Loss: 0.6347
    Epoch [3/5], Loss: 0.5907
    Epoch [4/5], Loss: 0.5654
    Epoch [5/5], Loss: 0.5591
  [성공] 시퀀스 3 학습 완료. 가중치가 'dataset/elephant_dst_to_src/fb/seq3/weights.pt'에 저장되었습니다.

--------------------------------------------------
[Phase] 시퀀스 길이 5 단계 시작
--------------------------------------------------
  [안내] 학습 데이터 개수: 8600개
    Epoch [1/5], Loss: 0.5531
    Epoch [2/5], Loss: 0.5484
    Epoch [3/5], Loss: 0.5479
    Epoch [4/5], Loss: 0.5465
    Epoch [5/5], Loss: 0.5465
  [성공] 시퀀스 5 학습 완료. 가중치가 'dataset/elephant_dst_to_src/fb/seq5/weights.pt'에 저장되었습니다.

--------------------------------------------------
[Phase] 시퀀스 길이 10 단계 시작
--------------------------------------------------
  [안내] 학습 데이터 개수: 8600개
    Epoch [1/5], Loss: 0.5528
    Epoch [2/5